# Session 10 notebook: active strategies from passive ETFs

**Question.** An ETF that tracks an index is "passive". But choosing *which* ETFs to hold, and *how much* of each, is an active decision. Can you run the strategies from this course with a few cheap, liquid ETFs? And when one looks good, is it the strategy, or something you chose with hindsight?

| Part | What you do | When |
|---|---|---|
| 1 | What do smart-beta ETFs actually deliver? Regress them on the factors | before class, 10 min |
| 2 | Volatility-managed equity from two ETFs (Moreira and Muir, implementable) | in class |
| 3 | Is it the timing, or the average holding? | in class |
| 4 | A three-ETF version, and the hindsight trap | in class |
| 5 | Discuss | in class |

**Data.** Everything downloads directly: ETF prices from Yahoo Finance (`yfinance`, already installed on Colab) and factors from Ken French's library. No WRDS needed, so you can reuse this code for your backtesting project.

Cells marked **Your turn** work as written. Change the value, rerun, and say what changed and why.

## Part 0. Setup (before class)

In [ ]:
import io, zipfile, urllib.request, itertools
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import yfinance as yf
pd.set_option("display.width", 160, "display.max_columns", 30, "display.float_format", "{:,.3f}".format)

KF = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/"

def kf(name, table=0):
    """Read one table from a Ken French CSV zip. Returns decimals, indexed by monthly Period or by day."""
    z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(KF + name + "_CSV.zip").read()))
    lines = z.read(z.namelist()[0]).decode("latin1").splitlines()
    i = [k for k, l in enumerate(lines) if l.startswith(",")][table]
    rows = []
    for l in lines[i + 1:]:
        if not l.strip() or not l.strip()[0].isdigit():
            break
        rows.append(l)
    df = pd.read_csv(io.StringIO("\n".join([lines[i]] + rows)), index_col=0)
    df.columns = [c.strip() for c in df.columns]
    idx = df.index.astype(str).str.strip()
    df.index = (pd.to_datetime(idx, format="%Y%m%d") if len(idx[0]) == 8
                else pd.PeriodIndex(idx.str[:4] + "-" + idx.str[4:], freq="M"))
    return df / 100

def stats(r, rf):
    """Annualized statistics for daily returns r; rf is the daily T-bill return."""
    nav = (1 + r).cumprod()
    ex = r - rf.reindex(r.index).fillna(0)
    return pd.Series({"CAGR": nav.iloc[-1] ** (252 / len(r)) - 1, "Vol": r.std() * np.sqrt(252),
                      "Sharpe": ex.mean() / r.std() * np.sqrt(252), "MaxDD": (nav / nav.cummax() - 1).min()})

def simulate(targets, R, tc_bps=2.0):
    """Daily returns of a portfolio rebalanced to `targets` (rows = rebalance dates, columns = ETFs) at each
    rebalance date's close. Weights drift with prices in between. Each rebalance pays tc_bps per dollar traded."""
    rows, w = [], None
    for d in R.index[R.index > targets.index[0]]:
        if w is None:
            w = targets.iloc[0].values.copy()
        r = R.loc[d].values
        port, cost = w @ r, 0.0
        w = w * (1 + r) / (1 + port)
        if d in targets.index:
            new = targets.loc[d].values
            cost = np.abs(new - w).sum() * tc_bps / 1e4
            w = new.copy()
        rows.append((d, port - cost))
    return pd.Series(dict(rows))

print("ready")

## Part 1. What do smart-beta ETFs actually deliver? (before class)

Four of the largest US "factor" ETFs, each built to tilt toward one of the anomalies from Weeks 1&ndash;3. Each is passive in form: it tracks an index. Each is active in function: it holds the market in very different weights.

We regress each ETF's monthly excess return on the Fama&ndash;French five factors plus momentum, from the month after it listed. This is Session 9's regression from Jones, applied to a product you can buy today.

- **Loadings**: did the ETF deliver the exposure on its label?
- **Alpha**: anything left over after the exposures?
- **Tracking error vs VTI**: how far from the market it strays, a rough measure of how "active" it is.

In [ ]:
SMART_BETA = {"USMV": "min volatility", "MTUM": "momentum", "VLUE": "value", "QUAL": "quality"}
FACTORS = ["Mkt-RF", "SMB", "HML", "RMW", "CMA", "Mom"]

px = yf.download(list(SMART_BETA) + ["VTI"], start="2011-01-01", auto_adjust=True, progress=False)["Close"]
m = px.resample("ME").last().pct_change()
m.index = m.index.to_period("M")
F = kf("F-F_Research_Data_5_Factors_2x3").join(kf("F-F_Momentum_Factor")).dropna()
print("Factors through", F.index[-1])

rows = {}
for t, label in SMART_BETA.items():
    y = (m[t] - F["RF"]).dropna()
    X = sm.add_constant(F.loc[y.index, FACTORS])
    fit = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": 3})
    rows[f"{t} ({label})"] = {"from": str(y.index[0]), "alpha, % a yr": fit.params["const"] * 1200,
                              "t(alpha)": fit.tvalues["const"], **fit.params[FACTORS].to_dict(), "R2": fit.rsquared,
                              "TE vs VTI": (m[t] - m["VTI"]).loc[y.index].std() * np.sqrt(12)}
part1 = pd.DataFrame(rows).T
part1

**Check yourself.**
1. Which ETF delivers the clearest version of its label? Which loading tells you?
2. USMV has a market beta well below 1. Over a rising market, what does that do to its return, even with zero alpha?
3. VLUE loads positively on HML but negatively on RMW (profitability). In words, what kind of "cheap" stocks is it buying? (Compare Glushkov's "unintended tilts".)
4. None of the alphas is significant. Is that a failure of these products?

## Part 2. Volatility-managed equity from two ETFs

Session 8's Moreira and Muir strategy, in a form you could actually run: hold **VTI** (the whole US market) and **VFISX** (Vanguard short-term Treasuries) as the safe asset. Each month-end:

- forecast VTI's volatility with an exponentially weighted average of past squared daily returns (&lambda; = 0.94, the RiskMetrics choice);
- put `TARGET / forecast` of the portfolio in VTI, capped at 100% (no leverage), and the rest in VFISX.

Two differences from Session 8: the target is a fixed number chosen in advance, not a constant fitted on the whole sample; and the costs are an ETF's costs, about 2 basis points per dollar traded.

In [ ]:
P = yf.download(["VTI", "QQQ", "VFISX", "VBMFX", "^IRX"], start="2001-01-01", auto_adjust=True, progress=False)["Close"]
rf = (P["^IRX"] / 100 / 252).ffill()                      # daily T-bill return
R = P[["VTI", "QQQ", "VFISX", "VBMFX"]].pct_change().dropna()
rf = rf.reindex(R.index).ffill()
month_end = R.index[R.groupby(R.index.to_period("M")).cumcount(ascending=False) == 0]
print("ETF data:", R.index[0].date(), "to", R.index[-1].date())

def vol_managed_targets(R, target, lam=0.94, cap=1.0, risky="VTI", safe="VFISX"):
    a, v, out = 1 - lam, R[risky].iloc[:63].var(), {}
    for d, x in R[risky].items():
        v = (1 - a) * v + a * x * x                       # EWMA variance, using data up to d only
        if d in month_end:
            w = min(cap, target / np.sqrt(v * 252))
            out[d] = {risky: w, safe: 1 - w}
    return pd.DataFrame(out).T.reindex(columns=R.columns, fill_value=0.0).iloc[3:]

**Your turn.** Change `TARGET` (try 0.10 and 0.15), `LAMBDA` (0.97 reacts more slowly) or `CAP` (1.5 means borrowing up to 50%).

In [ ]:
TARGET, LAMBDA, CAP, TC_BPS = 0.12, 0.94, 1.0, 2.0

W_vm = vol_managed_targets(R, TARGET, LAMBDA, CAP)
vm = simulate(W_vm, R, TC_BPS)
idx = vm.index
fixed = lambda w: pd.DataFrame([w] * len(W_vm), index=W_vm.index).reindex(columns=R.columns, fill_value=0.0)
b6040 = simulate(fixed({"VTI": 0.6, "VBMFX": 0.4}), R, TC_BPS)

part2 = pd.DataFrame({f"VTI vol-managed ({TARGET:.0%})": stats(vm, rf), "VTI": stats(R["VTI"].loc[idx], rf),
                      "60/40 (VTI / VBMFX)": stats(b6040.loc[idx], rf)}).T
print(f"Sample {idx[0].date()} to {idx[-1].date()}; average VTI weight {W_vm['VTI'].mean():.0%}; "
      f"average monthly change in the VTI weight {W_vm['VTI'].diff().abs().mean():.0%}")
part2

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(10, 7), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
pd.DataFrame({"vol-managed": (1 + vm).cumprod(), "VTI": (1 + R["VTI"].loc[idx]).cumprod(),
              "60/40": (1 + b6040.loc[idx]).cumprod()}).plot(ax=ax[0], logy=True, title="Growth of $1")
W_vm["VTI"].plot(ax=ax[1], title="Weight in VTI, set each month-end", ylim=(0, 1.05))
plt.tight_layout(); plt.show()

for a, b, name in [("2008-09-01", "2009-03-09", "2008 crisis"), ("2020-02-19", "2020-03-23", "Covid crash"),
                   ("2022-01-03", "2022-10-12", "2022 bear market")]:
    print(f"{name:17s} vol-managed {(1 + vm.loc[a:b]).prod() - 1:7.1%}   VTI {(1 + R['VTI'].loc[a:b]).prod() - 1:7.1%}"
          f"   VTI weight going in {W_vm['VTI'].asof(pd.Timestamp(a)):.0%}")

**Check yourself.**
1. In 2008 the strategy had already cut back before the worst months. In March 2020 it hadn't. Why not? (Look at the weight going in, and at how often the strategy trades.)
2. The strategy's turnover is high by Session 8's standards. Why do costs barely matter here?

## Part 3. Is it the timing, or the average holding?

The vol-managed portfolio holds about 80% VTI on average. Part 2 compared it with 100% VTI, which takes more risk. A fair comparison holds the **same average weights, fixed**. Then the only difference is the timing.

Three tests:
1. **Same average weights, fixed.**
2. **Spanning regression** (Session 8): regress the managed portfolio's monthly excess return on VTI's. A positive intercept means the timing adds something VTI can't replicate.
3. **Two halves of the sample.** Does the result hold in both?

In [ ]:
avg = W_vm.mean()
static = simulate(fixed(avg.to_dict()), R, TC_BPS)
part3 = pd.DataFrame({"vol-managed": stats(vm, rf), f"fixed {avg['VTI']:.0%} VTI / {avg['VFISX']:.0%} VFISX": stats(static.loc[idx], rf),
                      "VTI": stats(R["VTI"].loc[idx], rf)}).T
display(part3)

monthly = lambda r: (1 + r).groupby(r.index.to_period("M")).prod() - 1
rfm = monthly(rf.loc[idx])
fit = sm.OLS(monthly(vm) - rfm, sm.add_constant(monthly(R["VTI"].loc[idx]) - rfm)).fit(cov_type="HAC", cov_kwds={"maxlags": 3})
print(f"Spanning regression: alpha {fit.params['const'] * 1200:.2f}% a year (t = {fit.tvalues['const']:.2f}), beta {fit.params.iloc[1]:.2f}")

halves = {}
for a, b in [("2001", "2012"), ("2013", "2026")]:
    halves[f"{a}-{b}"] = {"vol-managed": stats(vm.loc[a:b], rf)["Sharpe"], "fixed weights": stats(static.loc[idx].loc[a:b], rf)["Sharpe"],
                          "VTI": stats(R["VTI"].loc[idx].loc[a:b], rf)["Sharpe"]}
print("\nSharpe ratio by half:")
pd.DataFrame(halves).T

**Check yourself.**
1. Over the full sample, how much of the Sharpe-ratio gain over VTI comes from the timing, and how much from simply holding less equity?
2. In which half does the timing work? What was different about volatility in that half? (Cederburg et al.'s point from Session 8, in ETF form.)
3. **Your turn.** Rerun Part 2 with `TC_BPS = 25`. Does the conclusion change?

## Part 4. A three-ETF version, and the hindsight trap

The same idea with more ingredients: VTI, **QQQ** (the Nasdaq-100) and VFISX, targeting 15% volatility with no leverage. Each month-end, among all mixes on a 5% grid whose forecast volatility is within 0.25% of the target, take the one closest to the current weights, which keeps trading low. If no mix hits the target, take the closest.

In [ ]:
ASSETS, TARGET3, STEP, TOL = ["VTI", "QQQ", "VFISX"], 0.15, 0.05, 0.0025

k = int(round(1 / STEP))
grid = np.array([c for c in itertools.product(range(k + 1), repeat=len(ASSETS)) if sum(c) == k]) / k
a, X = 1 - LAMBDA, R[ASSETS].values
S, cur, out, missed = np.cov(X[:63].T, bias=True), np.array([1.0, 0, 0]), {}, {}
for t, d in enumerate(R.index):
    S = (1 - a) * S + a * np.outer(X[t], X[t])           # EWMA covariance, data up to d only
    if d in month_end and t > 63:
        vols = np.sqrt(np.einsum("ij,jk,ik->i", grid, S * 252, grid))
        ok = np.abs(vols - TARGET3) <= TOL
        missed[d] = not ok.any()
        cand = np.where(ok)[0] if ok.any() else [np.argmin(np.abs(vols - TARGET3))]
        cur = grid[min(cand, key=lambda i: np.abs(grid[i] - cur).sum())]
        out[d] = dict(zip(ASSETS, cur))
W3 = pd.DataFrame(out).T.reindex(columns=R.columns, fill_value=0.0).iloc[3:]
v3 = simulate(W3, R, TC_BPS); i3 = v3.index
avg3 = W3.mean()
static3 = simulate(pd.DataFrame([avg3.values] * len(W3), index=W3.index, columns=R.columns), R, TC_BPS)

print("Average weights:", avg3[ASSETS].round(2).to_dict(), f"| months the target was out of reach: {pd.Series(missed).mean():.0%}")
pd.DataFrame({"3-ETF vol target": stats(v3, rf), "same average weights, fixed": stats(static3.loc[i3], rf),
              "VTI": stats(R["VTI"].loc[i3], rf), "QQQ": stats(R["QQQ"].loc[i3], rf)}).T

**Check yourself.**
1. The strategy beats VTI. How much of that is the timing, and how much is the average 70% in QQQ?
2. Why was QQQ in the menu? Would you have picked it in 2001, right after it fell 80%? This is the "smart beta mirage" in miniature: the ETF menu was chosen knowing which one won.
3. Why is the 15% target often out of reach? (Compare the three ETFs' volatilities.)
4. **Your turn.** Replace QQQ with a fund you would have chosen in 2001 without hindsight, such as `VBR` (small-cap value, from 2004) or `VEU` (international, from 2007). Rerun Parts 2&ndash;4 from the same start date.

## Part 5. Discuss

1. Part 1's ETFs are "passive" products with active-share-like tracking errors of 3&ndash;9% a year. In Easley et al.'s language, are they passive or active? Does the label matter for an investor?
2. Every result in Parts 2&ndash;4 rests on choices made in advance: the target, &lambda;, the cap, the ETF menu, the sample. Which of them could you defend *before* seeing the results?
3. Ben-David et al. find that thematic ETFs launch after their stocks have run up, and then lag. If you were launching a "volatility-managed" ETF today, what does the 2013&ndash;2026 half of Part 3 tell you to expect?
4. For your backtesting project: can your strategy be run with ETFs? If not, what would it cost to run with individual stocks (Session 8)?